# Fold-Specific Normalization and Classical Baseline

This notebook is the quality checkpoint before a CNN. It loads the validated 5-second windows, computes summary features, fits normalization only on training participants within each fold, and evaluates a class-weighted logistic-regression baseline at participant level.

A baseline that performs above chance gives the later CNN a meaningful reference. It also exposes whether performance is driven by window count, dataset identity, or a small number of participants.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
ARRAY_PATH = PROCESSED / 'validated_gait_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = PROJECT_ROOT / 'data' / 'interim' / 'participant_splits.csv'

windows = np.load(ARRAY_PATH, mmap_mode='r')
metadata = pd.read_csv(METADATA_PATH)
splits = pd.read_csv(SPLITS_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
assert len(metadata) == windows.shape[0]
assert metadata['participant_key'].isin(splits['participant_key']).all()
print('Window array:', windows.shape)
print('Windowed participants:', metadata['participant_key'].nunique())
print('Window counts by dataset and label:')
print(metadata.groupby(['dataset_id', 'label']).size())


Window array: (18511, 500, 18)
Windowed participants: 284
Window counts by dataset and label:
dataset_id    label  
felius_2024   healthy     2921
              stroke     13445
voisard_2025  healthy     1039
              stroke      1106
dtype: int64


In [2]:
def summary_features(window_array, batch_size=1024):
    features = []
    for start in range(0, len(window_array), batch_size):
        batch = np.asarray(window_array[start:start + batch_size], dtype=np.float32)
        means = batch.mean(axis=1)
        stds = batch.std(axis=1)
        rms = np.sqrt(np.mean(batch ** 2, axis=1))
        features.append(np.concatenate([means, stds, rms], axis=1))
    return np.concatenate(features, axis=0)


X_summary = summary_features(windows)
print('Summary feature matrix:', X_summary.shape)
assert np.isfinite(X_summary).all()


Summary feature matrix: (18511, 54)


## Participant-weighted fold evaluation

Training windows are weighted so each participant contributes equally, preventing long trials from dominating the baseline. Predictions are then averaged across windows for each participant before calculating metrics.

In [3]:
fold_results = []
participant_predictions = []
normalization_records = []

for fold in sorted(splits['fold'].unique()):
    fold_roles = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    roles = metadata['participant_key'].map(fold_roles)
    train_mask = roles.eq('training')
    validation_mask = roles.eq('validation')
    train_ids = metadata.index[train_mask].to_numpy()
    validation_ids = metadata.index[validation_mask].to_numpy()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_summary[train_ids])
    X_validation = scaler.transform(X_summary[validation_ids])
    y_train = metadata.loc[train_mask, 'label_binary'].to_numpy()
    y_validation = metadata.loc[validation_mask, 'label_binary'].to_numpy()

    participant_window_counts = metadata.loc[train_mask].groupby('participant_key').size()
    sample_weights = metadata.loc[train_mask, 'participant_key'].map(1.0 / participant_window_counts).to_numpy()
    sample_weights = sample_weights / sample_weights.mean()

    model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    validation_probabilities = model.predict_proba(X_validation)[:, 1]

    validation_frame = metadata.loc[validation_mask, ['participant_key', 'label', 'label_binary', 'dataset_id']].copy()
    validation_frame['probability'] = validation_probabilities
    participant_frame = validation_frame.groupby(['participant_key', 'label', 'label_binary', 'dataset_id'], as_index=False)['probability'].mean()
    participant_frame['fold'] = fold
    participant_predictions.append(participant_frame)

    y_true = participant_frame['label_binary'].to_numpy()
    y_prob = participant_frame['probability'].to_numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    fold_results.append({
        'fold': fold,
        'validation_participants': len(participant_frame),
        'healthy_participants': int((y_true == 0).sum()),
        'stroke_participants': int((y_true == 1).sum()),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'average_precision': average_precision_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
    })

    normalization_records.extend([
        {'fold': fold, 'statistic': 'mean', 'channel_feature': i, 'value': value}
        for i, value in enumerate(scaler.mean_)
    ])
    normalization_records.extend([
        {'fold': fold, 'statistic': 'scale', 'channel_feature': i, 'value': value}
        for i, value in enumerate(scaler.scale_)
    ])

fold_results = pd.DataFrame(fold_results)
participant_predictions = pd.concat(participant_predictions, ignore_index=True)
normalization_records = pd.DataFrame(normalization_records)
print(fold_results.round(3).to_string(index=False))
print()
print('Mean metrics:')
print(fold_results[['balanced_accuracy', 'roc_auc', 'average_precision', 'f1']].mean().round(3))


 fold  validation_participants  healthy_participants  stroke_participants  balanced_accuracy  roc_auc  average_precision    f1
    0                       57                    21                   36              0.952    0.999              0.999 0.973
    1                       58                    22                   36              0.895    0.938              0.934 0.933
    2                       57                    21                   36              0.962    0.993              0.996 0.972
    3                       56                    21                   35              0.948    0.967              0.984 0.957
    4                       56                    21                   35              0.957    0.995              0.997 0.955

Mean metrics:
balanced_accuracy    0.943
roc_auc              0.978
average_precision    0.982
f1                   0.958
dtype: float64


In [4]:
print('Validation performance by held-out dataset:')
dataset_results = []
for (fold, dataset_id), frame in participant_predictions.groupby(['fold', 'dataset_id']):
    if frame['label_binary'].nunique() < 2:
        continue
    y_true = frame['label_binary'].to_numpy()
    y_prob = frame['probability'].to_numpy()
    dataset_results.append({
        'fold': fold,
        'dataset_id': dataset_id,
        'participants': len(frame),
        'balanced_accuracy': balanced_accuracy_score(y_true, (y_prob >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y_true, y_prob),
    })
dataset_results = pd.DataFrame(dataset_results)
print(dataset_results.round(3).to_string(index=False))

windowed_participants = set(metadata['participant_key'])
all_participants = set(splits['participant_key'])
print()
print('Participants without valid windows:', len(all_participants - windowed_participants))


Validation performance by held-out dataset:
 fold   dataset_id  participants  balanced_accuracy  roc_auc
    0  felius_2024            38              0.944    0.996
    0 voisard_2025            19              0.958    1.000
    1  felius_2024            36              0.929    0.872
    1 voisard_2025            22              0.829    0.962
    2  felius_2024            31              0.917    0.987
    2 voisard_2025            26              0.955    1.000
    3  felius_2024            33              1.000    1.000
    3 voisard_2025            23              0.862    0.908
    4  felius_2024            25              1.000    1.000
    4 voisard_2025            31              0.893    0.996

Participants without valid windows: 4


In [5]:
baseline_dir = PROJECT_ROOT / 'data' / 'processed'
fold_results.to_csv(baseline_dir / 'classical_baseline_fold_results.csv', index=False)
participant_predictions.to_csv(baseline_dir / 'classical_baseline_participant_predictions.csv', index=False)
dataset_results.to_csv(baseline_dir / 'classical_baseline_dataset_results.csv', index=False)
normalization_records.to_csv(baseline_dir / 'fold_normalization_summary_feature_stats.csv', index=False)
print('Baseline outputs written to data/processed/')


Baseline outputs written to data/processed/


## Interpretation gate

Use the participant-level mean ROC-AUC and balanced accuracy as the reference for the first CNN. If the baseline is near chance, inspect data harmonization and labels before increasing model complexity. If it is clearly above chance but unstable across held-out datasets, treat dataset shift as the main next problem and report cross-dataset results separately.